# Tutorial 1: Synthetic Data Generation

## From Physical Parameters to Pulsar Timing Residuals

---

This notebook covers the **forward model** — the generative process that produces synthetic pulsar timing data from physical parameters. Understanding this stage is essential because the entire SBI pipeline rests on the fidelity and diversity of the training simulations.

### What you'll learn

1. How realistic observing schedules are generated (irregular cadence, seasonal gaps, heteroskedastic noise)
2. The Fourier-basis red and DM noise model and the enterprise power-law convention
3. How the factorized prior over 4 global + 3 per-backend white-noise parameters shapes the training distribution
4. Quasi-random sampling strategies for better prior coverage
5. The interplay between amplitude, spectral index, and data quality

### Modules covered

| Module | Purpose |
|--------|---------|
| `src.schedules` | Synthetic observing schedule generator |
| `src.priors` | `FactorizedPrior` (4D global + 3D WN per backend) |
| `src.simulator` | Fourier-basis red/DM noise + per-backend WN simulator |

In [ ]:
import sys, os

# Ensure the project root is on the path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

SEED = 42
print(f'NumPy {np.__version__}')

---
## 1. Observing Schedules

Real pulsar timing arrays observe each pulsar irregularly: telescopes have seasonal availability, cadence varies with observing campaigns, and some years may lack data entirely. Our schedule generator (`src.schedules.generate_schedule`) replicates these features:

- **Baseline**: 5–15 years of observations
- **Seasonal windows**: ~8 months on / ~4 months off per year, with ~20% chance of dropping an entire season
- **Cadence**: 2–6 week mean gaps within each season, randomly thinned
- **TOA count**: 80–400 observations per pulsar
- **Heteroskedastic noise**: $\sigma_i \sim 10^{\mathcal{U}(-7, -5)}$ seconds (100 ns – 10 μs)
- **Multi-frequency**: Observations at 820, 1400, and 2300 MHz
- **Backend instruments**: 1–3 receiver systems per pulsar

This diversity is critical — the neural network must learn to handle any combination of these properties.

In [ ]:
from src.schedules import generate_schedule

# Generate several schedules and inspect their properties
fig, axes = plt.subplots(4, 1, figsize=(11, 8), sharex=False)

for i, ax in enumerate(axes):
    rng = np.random.default_rng(SEED + i)
    sched = generate_schedule(rng)
    
    # Color by observing frequency
    colors = {820.0: 'C0', 1400.0: 'C1', 2300.0: 'C2'}
    for freq_val in np.unique(sched.freq_mhz):
        mask = sched.freq_mhz == freq_val
        ax.scatter(sched.t[mask], sched.sigma[mask], s=8, alpha=0.6,
                   c=colors.get(freq_val, 'gray'), label=f'{freq_val:.0f} MHz')
    
    ax.set_ylabel('$\\sigma_i$ (s)')
    ax.set_yscale('log')
    ax.set_title(f'Schedule {i+1}: N={sched.n_toa}, T={sched.tspan:.1f} yr, '
                 f'Backends={len(np.unique(sched.backend_id))}', fontsize=10)
    if i == 0:
        ax.legend(fontsize=8, ncol=3, loc='upper right')

axes[-1].set_xlabel('Time (years)')
fig.suptitle('Synthetic Observing Schedules', fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
# Statistical overview: generate many schedules and look at distributions
n_sched = 500
n_toas, tspans, mean_sigs, n_backends = [], [], [], []

for i in range(n_sched):
    rng = np.random.default_rng(10000 + i)
    s = generate_schedule(rng)
    n_toas.append(s.n_toa)
    tspans.append(s.tspan)
    mean_sigs.append(np.mean(s.sigma))
    n_backends.append(len(np.unique(s.backend_id)))

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
axes[0].hist(n_toas, bins=30, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Number of TOAs'); axes[0].set_ylabel('Count')
axes[0].set_title('TOA count distribution')

axes[1].hist(tspans, bins=30, color='coral', edgecolor='white')
axes[1].set_xlabel('Baseline (yr)'); axes[1].set_title('Baseline distribution')

axes[2].hist(np.log10(mean_sigs), bins=30, color='mediumseagreen', edgecolor='white')
axes[2].set_xlabel('$\\log_{10}\\langle\\sigma\\rangle$ (s)'); axes[2].set_title('Mean uncertainty')

axes[3].hist(n_backends, bins=[0.5, 1.5, 2.5, 3.5], color='mediumpurple', edgecolor='white', rwidth=0.7)
axes[3].set_xlabel('# Backends'); axes[3].set_title('Backends per pulsar')

fig.suptitle(f'Schedule statistics over {n_sched} realisations', fontsize=12, y=1.03)
fig.tight_layout()
plt.show()

print(f'TOAs: {np.mean(n_toas):.0f} ± {np.std(n_toas):.0f}  (range {min(n_toas)}–{max(n_toas)})')
print(f'Baseline: {np.mean(tspans):.1f} ± {np.std(tspans):.1f} yr')

---
## 2. The Prior: Sampling the Parameter Space

The prior $p(\boldsymbol{\theta})$ defines which parameters the simulator is asked to generate. The v5 **factorized prior** separates 7 parameters into two groups:

**Global parameters** (shared properties of the pulsar's noise environment):

| Parameter | Prior | Range |
|-----------|-------|-------|
| $\log_{10} A_{\rm red}$ | Uniform | $[-18, -11]$ |
| $\gamma_{\rm red}$ | Uniform | $[1.0, 7.0]$ |
| $\log_{10} A_{\rm dm}$ | Uniform | $[-18, -11]$ |
| $\gamma_{\rm dm}$ | Uniform | $[1.0, 7.0]$ |

**Per-backend white-noise parameters** (independent draw for each of up to 4 receiver systems):

| Parameter | Prior | Range |
|-----------|-------|-------|
| EFAC | Uniform | $[0.5, 2.0]$ |
| $\log_{10}$ EQUAD | Uniform | $[-8, -5]$ |
| $\log_{10}$ ECORR | Uniform | $[-8, -5]$ |

The `FactorizedPrior` class samples global and per-backend WN parameters independently, and supports both IID random and quasi-random sampling strategies.

In [ ]:
from src.priors import FactorizedPrior

global_bounds = {
    'log10_A_red': [-18, -11], 'gamma_red': [1.0, 7.0],
    'log10_A_dm': [-18, -11],  'gamma_dm':  [1.0, 7.0],
}
wn_bounds = {'EFAC': [0.5, 2.0], 'log10_EQUAD': [-8, -5], 'log10_ECORR': [-8, -5]}
prior = FactorizedPrior(global_bounds, wn_bounds, n_backends_max=4)

print(f'Global parameter names: {prior.global_names}')
print(f'WN parameter names:     {prior.wn_names}')
print(f'n_backends_max: {prior.n_backends_max}')

# Compare IID vs quasi-random sampling for the 4D global subspace
n_samples = 256
rng_iid = np.random.default_rng(SEED)
samples_global_iid  = prior.sample_global(n_samples, rng=rng_iid).numpy()    # (256, 4)
samples_global_qrng = prior.sample_global_sobol(n_samples, seed=SEED).numpy() # (256, 4)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

for ax, samps, label, color in [
    (axes[0], samples_global_iid,  f'IID Random (n={n_samples})',           'steelblue'),
    (axes[1], samples_global_qrng, f'Quasi-Random / Sobol (n={n_samples})', 'coral'),
]:
    ax.scatter(samps[:, 0], samps[:, 1], s=12, alpha=0.6, c=color)
    ax.set_title(label, fontsize=11)
    ax.set_xlabel('$\\log_{10} A_{\\rm red}$')
    ax.set_ylabel('$\\gamma_{\\rm red}$')
    ax.set_xlim(-18, -11); ax.set_ylim(1.0, 7.0)

fig.suptitle('Global Prior Sampling (log10_A_red vs gamma_red projection)', fontsize=12, y=1.02)
fig.tight_layout()
plt.show()

print('Quasi-random sampling fills the 4D global space more uniformly.')
print('This translates to better training coverage, especially at the prior boundaries.')

---
## 3. The Power-Law Spectrum (Enterprise Convention)

Red noise is modelled as a stochastic process with a **power-law spectral density**. We decompose the signal into $K$ Fourier modes with frequencies $f_k = k / T_{\rm span}$.

The per-mode Fourier-coefficient variance follows the standard PTA enterprise convention:

$$
\rho_k = \frac{A^2}{12\pi^2} \; T_{\rm yr}^2 \left(\frac{f_k}{f_{\rm ref}}\right)^{-\gamma} \Delta f
$$

where:
- $A = 10^{\log_{10} A_{\rm red}}$ is the dimensionless strain amplitude
- $T_{\rm yr} = 365.25 \times 86400$ s converts yr$^{-1}$ frequencies to seconds
- $f_{\rm ref} = 1$ yr$^{-1}$ is the reference frequency
- $\Delta f = 1/T_{\rm span}$ is the frequency resolution

The $1/(12\pi^2)$ normalisation connects the one-sided power spectral density to the strain amplitude $A$.

In [ ]:
from src.simulator import power_law_spectrum, SEC_PER_YR

print(f'SEC_PER_YR = {SEC_PER_YR:.2f} s')

tspan = 10.0  # years
n_modes = 30
freqs = np.arange(1, n_modes + 1) / tspan

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Vary amplitude
gammas_fixed = 3.5
for log10_A in [-16, -15, -14, -13, -12]:
    rho = power_law_spectrum(n_modes, tspan, log10_A, gammas_fixed)
    axes[0].loglog(freqs, rho, label=f'$\\log_{{10}} A = {log10_A}$', lw=1.5)
axes[0].set_xlabel('Frequency (yr$^{-1}$)')
axes[0].set_ylabel('$\\rho_k$ (s$^2$)')
axes[0].set_title(f'Varying amplitude ($\\gamma={gammas_fixed}$)')
axes[0].legend(fontsize=8)

# Vary spectral index
A_fixed = -14
for gamma in [1.0, 2.0, 3.5, 5.0, 6.5]:
    rho = power_law_spectrum(n_modes, tspan, A_fixed, gamma)
    axes[1].loglog(freqs, rho, label=f'$\\gamma = {gamma}$', lw=1.5)
axes[1].set_xlabel('Frequency (yr$^{-1}$)')
axes[1].set_ylabel('$\\rho_k$ (s$^2$)')
axes[1].set_title(f'Varying spectral index ($\\log_{{10}} A={A_fixed}$)')
axes[1].legend(fontsize=8)

fig.suptitle('Power-Law Spectrum (Enterprise Convention)', fontsize=12, y=1.02)
fig.tight_layout()
plt.show()

# Quantify mode dominance
print('\nFraction of total variance in fundamental mode (k=1):')
for gamma in [1.5, 3.5, 5.5]:
    rho = power_law_spectrum(n_modes, tspan, -14, gamma)
    print(f'  γ={gamma:.1f}: {rho[0]/rho.sum():.1%}  (ρ₁/ρ₂ = {rho[0]/rho[1]:.1f}×)')

---
## 4. The Fourier Design Matrix

The red noise signal is constructed via a **Fourier design matrix** $\mathbf{F}$. Given $N$ observation times and $K$ frequency modes:

$$
\mathbf{F}_{ij} = \begin{cases} \cos(2\pi f_k t_i) & \text{if } j = 2(k-1) \\ \sin(2\pi f_k t_i) & \text{if } j = 2(k-1)+1 \end{cases}
$$

The residuals are then:

$$
\mathbf{r} = \mathbf{F} \mathbf{a} + \boldsymbol{\varepsilon}, \qquad a_j \sim \mathcal{N}(0, \Phi_{jj}), \quad \varepsilon_i \sim \mathcal{N}(0, \sigma_i^2)
$$

where $\boldsymbol{\Phi} = \text{diag}(\rho_1, \rho_1, \rho_2, \rho_2, \ldots)$ repeats each $\rho_k$ twice (for sin and cos).

In [ ]:
from src.simulator import build_fourier_design_matrix

# Build F for a specific schedule
rng = np.random.default_rng(SEED)
sched = generate_schedule(rng)
tspan = sched.tspan
F = build_fourier_design_matrix(sched.t, tspan, n_modes=30)

print(f'Schedule: N={sched.n_toa} TOAs, T_span={tspan:.2f} yr')
print(f'Design matrix F: shape {F.shape}  (N_TOA × 2·n_modes)')

# Visualise the first few columns
fig, axes = plt.subplots(3, 2, figsize=(12, 6), sharex=True)

for k in range(3):
    fk = (k + 1) / tspan
    axes[k, 0].plot(sched.t, F[:, 2*k], '.', ms=2, alpha=0.6)
    axes[k, 0].set_ylabel(f'cos(2π·{fk:.2f}·t)')
    axes[k, 1].plot(sched.t, F[:, 2*k+1], '.', ms=2, alpha=0.6, color='C1')
    axes[k, 1].set_ylabel(f'sin(2π·{fk:.2f}·t)')
    if k == 0:
        axes[k, 0].set_title('Cosine columns')
        axes[k, 1].set_title('Sine columns')

axes[-1, 0].set_xlabel('Time (yr)')
axes[-1, 1].set_xlabel('Time (yr)')
fig.suptitle('Fourier Design Matrix F — first 3 modes sampled at TOA times', fontsize=12, y=1.02)
fig.tight_layout()
plt.show()

print('\nNote how the irregular sampling creates aliases — the Fourier modes')
print('are NOT orthogonal on the irregular grid (unlike uniform sampling).')
print(f'F^T F diagonal range: [{np.diag(F.T @ F).min():.1f}, {np.diag(F.T @ F).max():.1f}]')
print(f'max off-diagonal |F^T F|: {np.max(np.abs(F.T @ F - np.diag(np.diag(F.T @ F)))):.3f}')

---
## 5. Full Simulation: Parameters → Residuals

The `simulate_pulsar` function ties everything together: draw Fourier coefficients from the power-law spectrum, project through the design matrix, and add white noise. Let's see how the residuals change across parameter space.

In [ ]:
from src.simulator import simulate_pulsar

# Simulate at the four corners of an instructive parameter subspace
test_cases = [
    (np.array([-16.0, 2.0]), 'Weak amplitude, flat spectrum'),
    (np.array([-12.5, 2.0]), 'Strong amplitude, flat spectrum'),
    (np.array([-16.0, 5.5]), 'Weak amplitude, steep spectrum'),
    (np.array([-12.5, 5.5]), 'Strong amplitude, steep spectrum'),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 7))
axes = axes.ravel()

for i, (theta, label) in enumerate(test_cases):
    rng_i = np.random.default_rng(SEED + 100 + i)
    sched_i = generate_schedule(rng_i)
    sim = simulate_pulsar(theta, sched_i, n_modes=30, rng=rng_i)
    
    ax = axes[i]
    ax.errorbar(sim.t, sim.residuals, yerr=sim.sigma, fmt='.', ms=3,
                alpha=0.5, elinewidth=0.5, capsize=0)
    ax.axhline(0, color='gray', ls='--', lw=0.5)
    ax.set_title(f'{label}\n$\\log_{{10}} A={theta[0]:.1f}$, $\\gamma={theta[1]:.1f}$, '
                 f'N={len(sim.t)}, T={sim.tspan:.1f} yr', fontsize=9)
    ax.set_xlabel('Time (yr)')
    ax.set_ylabel('Residual (s)')

fig.suptitle('Simulated Residuals Across Parameter Space', fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

print('Key degeneracy: both increasing A and steepening γ produce')
print('more low-frequency power, making the two parameters correlated.')

In [ ]:
# Decompose one simulation into red noise vs white noise components
rng_decomp = np.random.default_rng(2024)
theta_demo = np.array([-13.5, 4.0])
sched_demo = generate_schedule(rng_decomp)

# Simulate step-by-step to get components
log10_A, gamma = theta_demo
F_demo = build_fourier_design_matrix(sched_demo.t, sched_demo.tspan, 30)
rho_demo = power_law_spectrum(30, sched_demo.tspan, log10_A, gamma)
phi_diag = np.repeat(rho_demo, 2)
a_coeffs = rng_decomp.normal(size=60).astype(np.float32) * np.sqrt(phi_diag)
red_noise = F_demo @ a_coeffs
white_noise = rng_decomp.normal(size=len(sched_demo.t)).astype(np.float32) * sched_demo.sigma
total = red_noise + white_noise

fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)

axes[0].plot(sched_demo.t, red_noise, '.', ms=3, alpha=0.6, color='C3')
axes[0].set_ylabel('Red noise (s)'); axes[0].set_title('Red noise component (Fa)', fontsize=10)

axes[1].plot(sched_demo.t, white_noise, '.', ms=3, alpha=0.6, color='C0')
axes[1].set_ylabel('White noise (s)'); axes[1].set_title('White noise component (ε)', fontsize=10)

axes[2].errorbar(sched_demo.t, total, yerr=sched_demo.sigma, fmt='.', ms=3,
                 alpha=0.5, elinewidth=0.5, capsize=0, color='C2')
axes[2].set_ylabel('Total (s)'); axes[2].set_xlabel('Time (yr)')
axes[2].set_title('Observed residuals (red + white)', fontsize=10)

for ax in axes:
    ax.axhline(0, color='gray', ls='--', lw=0.5)

fig.suptitle(f'Signal Decomposition — $\\log_{{10}}A={log10_A}$, $\\gamma={gamma}$',
             fontsize=12, y=1.02)
fig.tight_layout()
plt.show()

snr_rms = np.std(red_noise) / np.mean(sched_demo.sigma)
print(f'Red noise RMS: {np.std(red_noise):.2e} s')
print(f'Mean white noise σ: {np.mean(sched_demo.sigma):.2e} s')
print(f'Approximate red/white SNR: {snr_rms:.1f}')

---
## 6. Sensitivity Across the Prior Volume

An important question for SBI training: **which regions of parameter space produce detectable signals?** If the amplitude is too weak relative to the white noise, the data carry almost no information about $A$ or $\gamma$, and the posterior will be close to the prior.

In [ ]:
# Compute approximate red-noise SNR across a grid of parameters
# SNR ~ std(red_noise) / mean(sigma)
# We estimate std(red_noise) from the trace of F @ diag(phi) @ F^T

rng_sens = np.random.default_rng(999)
sched_sens = generate_schedule(rng_sens, tspan_min_yr=10, tspan_max_yr=10,
                                n_toa_min=200, n_toa_max=200)
F_sens = build_fourier_design_matrix(sched_sens.t, sched_sens.tspan, 30)
mean_sigma = np.mean(sched_sens.sigma)

A_grid = np.linspace(-18, -11, 60)
G_grid = np.linspace(1.0, 7.0, 50)
snr_map = np.zeros((len(A_grid), len(G_grid)))

for i, logA in enumerate(A_grid):
    for j, gam in enumerate(G_grid):
        rho = power_law_spectrum(30, sched_sens.tspan, logA, gam)
        phi_diag = np.repeat(rho, 2)
        # Trace of F Phi F^T = sum of phi_k * ||F_k||^2
        red_var = np.sum(phi_diag * np.sum(F_sens**2, axis=0))
        snr_map[i, j] = np.sqrt(red_var / len(sched_sens.t)) / mean_sigma

fig, ax = plt.subplots(figsize=(8, 5.5))
AA, GG = np.meshgrid(A_grid, G_grid, indexing='ij')
c = ax.contourf(AA, GG, np.log10(snr_map + 1e-30), levels=20, cmap='viridis')
plt.colorbar(c, ax=ax, label='$\\log_{10}$ SNR')
ax.contour(AA, GG, snr_map, levels=[1.0], colors='red', linewidths=2, linestyles='--')
ax.text(-14.5, 2.0, 'SNR = 1', color='red', fontsize=10, fontweight='bold')
ax.set_xlabel('$\\log_{10} A_{\\rm red}$')
ax.set_ylabel('$\\gamma_{\\rm red}$')
ax.set_title('Approximate Red Noise SNR Across Prior Volume\n'
             f'(N=200 TOAs, T=10 yr, $\\langle\\sigma\\rangle$={mean_sigma:.1e} s)', fontsize=11)
plt.tight_layout()
plt.show()

print('Region below the red dashed line (SNR < 1) is where red noise is buried')
print('in white noise — the network must learn to produce broad posteriors there.')
print('Region above is where decisive parameter estimation is possible.')

---
## Summary

This notebook covered the three foundational components of the v5 data generation stage:

| Component | Module | Key Output |
|-----------|--------|------------|
| Observing schedule | `src.schedules` | Irregular times, $\sigma_i$, frequencies, backend IDs |
| Factorized prior | `src.priors.FactorizedPrior` | 4D global + 3D WN per-backend parameter draws |
| Full noise simulation | `src.simulator` | Fourier red/DM noise, per-backend EFAC/EQUAD/ECORR WN |

### Key takeaways

1. **Diversity is deliberate**: The schedule generator produces a wide range of observing conditions so the network generalises
2. **Quasi-random sampling** improves training coverage over naive IID sampling, especially in 4D global space
3. **The enterprise convention** ($1/12\pi^2 \cdot \text{yr}^2$ prefactor) makes strain amplitudes physically meaningful
4. **Mode dominance**: steep $\gamma$ concentrates variance in the fundamental mode — the network must learn this structure
5. **Detection horizon**: much of the prior volume has SNR < 1, where the posterior should revert to the prior
6. **Per-backend white noise** (EFAC, EQUAD, ECORR) adds realistic instrument-dependent noise scaling that must be inferred jointly

**Next**: [Tutorial 2 — The Data Pipeline](02_data_pipeline.ipynb) covers tokenization, masking augmentations, and batch collation.